# NLP Data Loader with PyTorch


**Author:** Hadeer Nabil  
**Title:** Senior Data Scientist & AI/LLM Engineer

## About this Notebook

This notebook demonstrates how to build a complete NLP data preprocessing pipeline using PyTorch. The workflow covers tokenization, vocabulary creation, numerical encoding, custom datasets, DataLoaders, padding of variable-length sequences, and preparation of bilingual datasets for machine translation tasks.

The resulting pipeline can be used as a foundation for training modern NLP systems such as:

- Sentiment Analysis Models
- Text Classification Models
- Machine Translation Systems
- Question Answering Systems
- Retrieval-Augmented Generation (RAG) Applications
- Large Language Models (LLMs)

## Technologies Used

- Python
- PyTorch
- TorchText
- NLP Preprocessing
- Data Loading & Batching
- Machine Translation Data Preparation

## Repository Purpose

This notebook is part of my Machine Learning, Generative AI, and LLM Engineering portfolio. It demonstrates the fundamental NLP data engineering concepts required before training deep learning and language models.

## 1. Imports

We use PyTorch for dataset and batch handling, and torchtext for tokenization and vocabulary creation.

In [1]:
import torch

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

from typing import Iterable, List, Tuple

## 2. Sample English Sentences

These sentences are used to demonstrate a basic NLP DataLoader.

In real projects, the text could come from CSV files, databases, scraped text, customer reviews, support tickets, or document chunks in a RAG pipeline.

In [2]:
sentences = [
    "If you want to know what a man's like, take a good look at how he treats his inferiors, not his equals.",
    "Fame's a fickle friend, Harry.",
    "It is our choices, Harry, that show what we truly are, far more than our abilities.",
    "Soon we must all face the choice between what is right and what is easy.",
    "Youth can not know how age thinks and feels. But old men are guilty if they forget what it was to be young.",
    "You are awesome!",
    "Machine learning models require high-quality training data.",
    "Natural language processing enables computers to understand human text.",
    "The application failed because the database connection timed out.",
    "Data quality is more important than data quantity."
]

## 3. Tokenization

Tokenization converts raw text into smaller units called tokens.

Example:

`"You are awesome!"`

becomes something like:

`["you", "are", "awesome", "!"]`

In [3]:
tokenizer = get_tokenizer("basic_english")

tokens = tokenizer(sentences[0])
print(tokens)

['if', 'you', 'want', 'to', 'know', 'what', 'a', 'man', "'", 's', 'like', ',', 'take', 'a', 'good', 'look', 'at', 'how', 'he', 'treats', 'his', 'inferiors', ',', 'not', 'his', 'equals', '.']


## 4. Build a Vocabulary

A vocabulary maps each token to an integer ID.

Neural networks do not understand raw text directly, so text must be converted into numbers first.

In [4]:
vocab = build_vocab_from_iterator(map(tokenizer, sentences), specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])

print("Vocabulary size:", len(vocab))
print("Token ID for 'you':", vocab["you"])
print("Token ID for '<pad>':", vocab["<pad>"])

Vocabulary size: 94
Token ID for 'you': 25
Token ID for '<pad>': 1


## 5. Custom Dataset Returning Token IDs

A PyTorch `Dataset` tells PyTorch how to access individual samples.

For NLP, each sentence is:

1. Tokenized
2. Converted to token IDs
3. Returned as a tensor

In [5]:
class CustomDataset(Dataset):
    def __init__(self, sentences, tokenizer, vocab):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.vocab = vocab

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        tokens = self.tokenizer(self.sentences[idx])
        token_ids = [self.vocab[token] for token in tokens]
        return torch.tensor(token_ids, dtype=torch.long)

In [6]:
custom_dataset = CustomDataset(sentences, tokenizer, vocab)

print("Dataset length:", len(custom_dataset))
print("First sample tensor:")
print(custom_dataset[0])

Dataset length: 10
First sample tensor:
tensor([16, 25, 90, 10, 18,  4,  6, 66, 11, 22, 63,  3, 80,  6, 54, 64, 31, 15,
        56, 87, 14, 60,  3, 20, 14, 45,  2])


## 6. Inspect Sequence Lengths

Different sentences have different numbers of tokens.

This is important because PyTorch's default DataLoader cannot stack tensors of different lengths into one batch.

In [7]:
for i in range(len(custom_dataset)):
    print(f"Item {i}: length = {len(custom_dataset[i])}")

Item 0: length = 27
Item 1: length = 9
Item 2: length = 20
Item 3: length = 16
Item 4: length = 25
Item 5: length = 4
Item 6: length = 8
Item 7: length = 10
Item 8: length = 10
Item 9: length = 9


## 7. Why the Default DataLoader Fails

The default `DataLoader` tries to stack samples into a batch using `torch.stack`.

That only works if all tensors have exactly the same shape.

Text sequences usually have different lengths, so the default batching often fails.

The code below catches the error so the notebook can continue running.

In [8]:
batch_size = 2

try:
    dataloader_without_padding = DataLoader(
        custom_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    for batch in dataloader_without_padding:
        print(batch)

except RuntimeError as error:
    print("Expected DataLoader error:")
    print(error)

Expected DataLoader error:
stack expects each tensor to be equal size, but got [10] at entry 0 and [25] at entry 1


## 8. Fix: Pad Sequences Inside the Batch

To batch variable-length sequences, shorter sequences are padded with the `<pad>` token.

For example:

```text
[10, 11, 12, 13]
[20, 21]
```

becomes:

```text
[10, 11, 12, 13]
[20, 21,  1,  1]
```

Here, `1` is the ID of the `<pad>` token.

In [9]:
pad_index = vocab["<pad>"]

# This function is needed because DataLoader needs instructions
# for how to combine variable-length tensors into one batch.
def collate_fn(batch):
    return pad_sequence(batch, batch_first=True, padding_value=pad_index)

## 9. DataLoader with Padding

Now the DataLoader can create batches successfully.

Each batch has shape:

`[batch_size, max_sequence_length_in_that_batch]`

In [10]:
dataloader = DataLoader(
    custom_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

for batch in dataloader:
    print(batch)
    print("Batch shape:", batch.shape)
    print("-" * 50)

tensor([[65, 62, 68, 76, 57, 86,  8,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1,  1,  1,  1,  1,  1],
        [93, 37, 20, 18, 15, 28, 84, 12, 50,  2, 36, 71, 67,  7, 55, 16, 83, 52,
          4, 17, 91, 10, 33, 92,  2]])
Batch shape: torch.Size([2, 25])
--------------------------------------------------
tensor([[79, 24, 69, 29, 46,  9, 38, 35,  4,  5, 77, 12,  4,  5, 43,  2],
        [70, 61, 73, 44, 40, 10, 89, 58, 81,  2,  1,  1,  1,  1,  1,  1]])
Batch shape: torch.Size([2, 16])
--------------------------------------------------
tensor([[ 8, 74,  5, 19, 59, 23,  8, 75,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1,  1,  1,  1,  1,  1,  1,  1],
        [16, 25, 90, 10, 18,  4,  6, 66, 11, 22, 63,  3, 80,  6, 54, 64, 31, 15,
         56, 87, 14, 60,  3, 20, 14, 45,  2]])
Batch shape: torch.Size([2, 27])
--------------------------------------------------
tensor([[48, 11, 22,  6, 51, 53,  3, 13,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1],
      

## 10. Decode Token IDs Back to Tokens

This helper function is useful for debugging.

It shows what a padded tensor represents as tokens.

In [11]:
index_to_token = vocab.get_itos()

def decode_token_ids(token_ids):
    return [index_to_token[token_id] for token_id in token_ids]

sample_batch = next(iter(dataloader))

for row in sample_batch:
    print(decode_token_ids(row.tolist()))

['the', 'application', 'failed', 'because', 'the', 'database', 'connection', 'timed', 'out', '.']
['fame', "'", 's', 'a', 'fickle', 'friend', ',', 'harry', '.', '<pad>']


## 11. Alternative Pattern: Dataset Returns Raw Text

Another common design is to let the Dataset return raw text, then tokenize and numericalize inside `collate_fn`.

This is useful when batching logic needs more control.

In [12]:
class RawTextDataset(Dataset):
    def __init__(self, sentences):
        self.sentences = sentences

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return self.sentences[idx]

In [13]:
def text_collate_fn(batch):
    tensor_batch = []

    for sentence in batch:
        tokens = tokenizer(sentence)
        token_ids = [vocab[token] for token in tokens]
        tensor_batch.append(torch.tensor(token_ids, dtype=torch.long))

    return pad_sequence(tensor_batch, batch_first=True, padding_value=pad_index)

In [14]:
raw_text_dataset = RawTextDataset(sentences)

raw_text_dataloader = DataLoader(
    raw_text_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=text_collate_fn
)

for batch in raw_text_dataloader:
    print(batch)
    print("Batch shape:", batch.shape)
    print("-" * 50)

tensor([[ 9, 30, 47, 34,  9, 42, 41, 85, 72,  2],
        [70, 61, 73, 44, 40, 10, 89, 58, 81,  2]])
Batch shape: torch.Size([2, 10])
--------------------------------------------------
tensor([[16, 25, 90, 10, 18,  4,  6, 66, 11, 22, 63,  3, 80,  6, 54, 64, 31, 15,
         56, 87, 14, 60,  3, 20, 14, 45,  2],
        [ 8, 74,  5, 19, 59, 23,  8, 75,  2,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1,  1,  1,  1,  1,  1,  1,  1]])
Batch shape: torch.Size([2, 27])
--------------------------------------------------
tensor([[65, 62, 68, 76, 57, 86,  8,  2,  1],
        [48, 11, 22,  6, 51, 53,  3, 13,  2]])
Batch shape: torch.Size([2, 9])
--------------------------------------------------
tensor([[79, 24, 69, 29, 46,  9, 38, 35,  4,  5, 77, 12,  4,  5, 43,  2,  1,  1,
          1,  1,  1,  1,  1,  1,  1],
        [93, 37, 20, 18, 15, 28, 84, 12, 50,  2, 36, 71, 67,  7, 55, 16, 83, 52,
          4, 17, 91, 10, 33, 92,  2]])
Batch shape: torch.Size([2, 25])
----------------------------

## 12. French Text Example

This example shows the same pipeline with French text.

The process is identical:

1. Create text corpus
2. Tokenize text
3. Build vocabulary
4. Create Dataset
5. Pad sequences in DataLoader

In [15]:
french_corpus = [
    "Ceci est une phrase.",
    "C'est un autre exemple de phrase.",
    "Voici une troisième phrase.",
    "Il fait beau aujourd'hui.",
    "J'aime beaucoup la cuisine française.",
    "Que fais-tu ce weekend?",
    "Nous apprenons le traitement automatique du langage naturel.",
    "Les données textuelles doivent être transformées en nombres."
]

In [16]:
french_tokenizer = get_tokenizer("basic_english")

french_vocab = build_vocab_from_iterator(
    map(french_tokenizer, french_corpus),
    specials=["<unk>", "<pad>"]
)
french_vocab.set_default_index(french_vocab["<unk>"])

french_pad_index = french_vocab["<pad>"]

In [17]:
class FrenchTextDataset(Dataset):
    def __init__(self, corpus):
        self.corpus = corpus

    def __len__(self):
        return len(self.corpus)

    def __getitem__(self, idx):
        return self.corpus[idx]

In [18]:
def french_collate_fn(batch):
    # Sort by sequence length, longest first.
    # This is useful for some RNN/LSTM workflows.
    batch = sorted(batch, key=lambda sentence: len(french_tokenizer(sentence)), reverse=True)

    tensor_batch = []

    for sentence in batch:
        tokens = french_tokenizer(sentence)
        token_ids = [french_vocab[token] for token in tokens]
        tensor_batch.append(torch.tensor(token_ids, dtype=torch.long))

    return pad_sequence(tensor_batch, batch_first=True, padding_value=french_pad_index)

In [19]:
french_dataset = FrenchTextDataset(french_corpus)

french_dataloader = DataLoader(
    french_dataset,
    batch_size=3,
    shuffle=False,
    collate_fn=french_collate_fn
)

for batch in french_dataloader:
    print(batch)
    print("Batch shape:", batch.shape)
    print("-" * 50)

tensor([[15,  3,  5, 43, 12, 24, 19,  4,  2],
        [17,  5,  6,  4,  2,  1,  1,  1,  1],
        [44,  6, 42,  4,  2,  1,  1,  1,  1]])
Batch shape: torch.Size([3, 9])
--------------------------------------------------
tensor([[30,  3,  8, 14, 31, 18, 27,  2],
        [29, 26, 13, 10,  3, 28,  2,  1],
        [38, 25, 16, 45,  7,  1,  1,  1]])
Batch shape: torch.Size([3, 8])
--------------------------------------------------
tensor([[37,  9, 33, 40, 11, 22, 32, 35,  2],
        [34, 21, 39, 20, 46, 41, 23, 36,  2]])
Batch shape: torch.Size([2, 9])
--------------------------------------------------


# Machine Translation Data Loader

For translation, each sample has two parts:

- Source sentence: input language, for example German
- Target sentence: output language, for example English

The DataLoader must return two padded tensors:

1. `src_batch`
2. `tgt_batch`

## 13. Small German-English Translation Dataset

To keep this notebook fully runnable without downloading external datasets, we use a small toy translation dataset.

In a real machine translation project, this could be replaced with Multi30k, WMT, OPUS, or another parallel corpus.

In [20]:
translation_pairs = [
    ("Ein Mann geht die Straße entlang.", "A man is walking down the street."),
    ("Eine Frau liest ein Buch.", "A woman is reading a book."),
    ("Das Kind spielt im Garten.", "The child is playing in the garden."),
    ("Zwei Hunde laufen durch den Park.", "Two dogs are running through the park."),
    ("Ich lerne maschinelles Lernen.", "I am learning machine learning."),
    ("Daten müssen vor dem Training vorbereitet werden.", "Data must be prepared before training."),
    ("Dieses Notebook zeigt einen NLP DataLoader.", "This notebook demonstrates an NLP DataLoader."),
    ("Wir erstellen Batches mit Padding.", "We create batches with padding.")
]

## 14. Translation Tokenizers and Special Tokens

Translation models often use special tokens:

- `<unk>`: unknown token
- `<pad>`: padding token
- `<bos>`: beginning of sentence
- `<eos>`: end of sentence

In [21]:
SRC_LANGUAGE = "de"
TGT_LANGUAGE = "en"

UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]

token_transform = {
    SRC_LANGUAGE: get_tokenizer("basic_english"),
    TGT_LANGUAGE: get_tokenizer("basic_english")
}

## 15. Build Source and Target Vocabularies

For translation, we usually maintain separate vocabularies for the source and target languages.

In [22]:
def yield_tokens(data_iter: Iterable[Tuple[str, str]], language: str):
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}

    for data_sample in data_iter:
        yield token_transform[language](data_sample[language_index[language]])

In [23]:
vocab_transform = {}

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[language] = build_vocab_from_iterator(
        yield_tokens(translation_pairs, language),
        min_freq=1,
        specials=special_symbols,
        special_first=True
    )
    vocab_transform[language].set_default_index(UNK_IDX)

print("German vocab size:", len(vocab_transform[SRC_LANGUAGE]))
print("English vocab size:", len(vocab_transform[TGT_LANGUAGE]))

German vocab size: 48
English vocab size: 46


## 16. Convert Text to Tensors

For each sentence, we apply:

1. Tokenization
2. Vocabulary lookup
3. Add `<bos>` and `<eos>` tokens
4. Convert to tensor

In [24]:
def tensor_transform(token_ids: List[int]):
    return torch.cat((
        torch.tensor([BOS_IDX], dtype=torch.long),
        torch.tensor(token_ids, dtype=torch.long),
        torch.tensor([EOS_IDX], dtype=torch.long)
    ))


def sequential_transforms(*transforms):
    def func(text_input):
        for transform in transforms:
            text_input = transform(text_input)
        return text_input
    return func

In [25]:
text_transform = {}

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[language] = sequential_transforms(
        token_transform[language],
        vocab_transform[language],
        tensor_transform
    )

In [26]:
example_de, example_en = translation_pairs[0]

print("German:", example_de)
print(text_transform[SRC_LANGUAGE](example_de))

print("English:", example_en)
print(text_transform[TGT_LANGUAGE](example_en))

German: Ein Mann geht die Straße entlang.
tensor([ 2,  5, 31, 22, 13, 40, 18,  4,  3])
English: A man is walking down the street.
tensor([ 2,  6, 27,  7, 42, 22,  5, 37,  4,  3])


## 17. Translation Dataset

The dataset returns a tuple:

`(source_sentence, target_sentence)`

In [27]:
class TranslationDataset(Dataset):
    def __init__(self, sentence_pairs):
        self.sentence_pairs = sentence_pairs

    def __len__(self):
        return len(self.sentence_pairs)

    def __getitem__(self, idx):
        return self.sentence_pairs[idx]

## 18. Translation Collate Function

This function prepares a batch of translation pairs.

It creates:

- A padded German source batch
- A padded English target batch

In [28]:
def translation_collate_fn(batch):
    src_batch = []
    tgt_batch = []

    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample))

    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)

    return src_batch, tgt_batch

## 19. Translation DataLoader

The translation DataLoader now returns two tensors per batch:

```python
src_batch, tgt_batch
```

In [29]:
translation_dataset = TranslationDataset(translation_pairs)

translation_dataloader = DataLoader(
    translation_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=translation_collate_fn
)

for src_batch, tgt_batch in translation_dataloader:
    print("Source batch:")
    print(src_batch)
    print("Source shape:", src_batch.shape)

    print("Target batch:")
    print(tgt_batch)
    print("Target shape:", tgt_batch.shape)
    print("-" * 60)

Source batch:
tensor([[ 2, 10, 34, 42, 11, 41, 43, 44,  4,  3],
        [ 2, 45, 19,  6, 33, 37,  4,  3,  1,  1]])
Source shape: torch.Size([2, 10])
Target batch:
tensor([[ 2, 18, 28, 13, 34, 14, 40,  4,  3],
        [ 2, 43, 17, 12, 44, 31,  4,  3,  1]])
Target shape: torch.Size([2, 9])
------------------------------------------------------------
Source batch:
tensor([[ 2, 47, 23, 27, 15, 12, 38,  4,  3],
        [ 2,  8, 26, 39, 25, 21,  4,  3,  1]])
Source shape: torch.Size([2, 9])
Target batch:
tensor([[ 2, 41, 21, 11, 36, 39,  5, 32,  4,  3],
        [ 2,  5, 16,  7, 33, 25,  5, 23,  4,  3]])
Target shape: torch.Size([2, 10])
------------------------------------------------------------
Source batch:
tensor([[ 2, 14, 36, 46, 17, 35,  9,  4,  3],
        [ 2, 24, 28, 32, 29,  4,  3,  1,  1]])
Source shape: torch.Size([2, 9])
Target batch:
tensor([[ 2, 38, 30, 20, 10, 29, 19,  4,  3],
        [ 2, 24,  9,  8, 26,  8,  4,  3,  1]])
Target shape: torch.Size([2, 9])
--------------------

## 20. Decode Translation Batches

This helper is useful for checking whether the source and target batches were created correctly.

In [31]:
def decode_sequence(sequence, vocab):
    tokens = vocab.get_itos()
    return [tokens[token_id] for token_id in sequence]

src_batch, tgt_batch = next(iter(translation_dataloader))

print("Decoded German source batch:")
for sequence in src_batch:
    print(decode_sequence(sequence.tolist(), vocab_transform[SRC_LANGUAGE]))

print("Decoded English target batch:")
for sequence in tgt_batch:
    print(decode_sequence(sequence.tolist(), vocab_transform[TGT_LANGUAGE]))

Decoded German source batch:
['<bos>', 'wir', 'erstellen', 'batches', 'mit', 'padding', '.', '<eos>', '<pad>', '<pad>']
['<bos>', 'daten', 'müssen', 'vor', 'dem', 'training', 'vorbereitet', 'werden', '.', '<eos>']
Decoded English target batch:
['<bos>', 'we', 'create', 'batches', 'with', 'padding', '.', '<eos>', '<pad>']
['<bos>', 'data', 'must', 'be', 'prepared', 'before', 'training', '.', '<eos>']


In [33]:
# Test the translation data pipeline with a custom German sentence

test_german_sentence = "ich liebe maschinelles lernen"

print("Original German sentence:")
print(test_german_sentence)

# Tokenize German sentence
tokens = token_transform[SRC_LANGUAGE](test_german_sentence)
print("\nTokenized German sentence:")
print(tokens)

# Convert tokens to IDs
token_ids = vocab_transform[SRC_LANGUAGE](tokens)
print("\nNumerical token IDs:")
print(token_ids)

# Convert to tensor
tensor_sentence = torch.tensor(token_ids, dtype=torch.long)
print("\nTensor representation:")
print(tensor_sentence)

# Decode back to tokens
decoded_tokens = decode_sequence(tensor_sentence.tolist(), vocab_transform[SRC_LANGUAGE])
print("\nDecoded back to tokens:")
print(decoded_tokens)

Original German sentence:
ich liebe maschinelles lernen

Tokenized German sentence:
['ich', 'liebe', 'maschinelles', 'lernen']

Numerical token IDs:
[24, 0, 32, 29]

Tensor representation:
tensor([24,  0, 32, 29])

Decoded back to tokens:
['ich', '<unk>', 'maschinelles', 'lernen']


# Summary

This notebook demonstrates the full NLP DataLoader workflow:

- Raw text cannot be used directly by neural networks.
- Text must be tokenized and converted into token IDs.
- Sentences usually have different lengths.
- Default PyTorch DataLoader fails when tensors have different lengths.
- `pad_sequence` fixes this by padding shorter sequences.
- For translation tasks, the DataLoader should return both source and target batches.

This notebook can be reused as a starting point for:

- Text classification
- Sentiment analysis
- RNN/LSTM/GRU models
- Transformer models
- Machine translation
- RAG document chunk batching